In [ ]:
# Задание 1
from Bio import Entrez, SeqIO
from Bio.SeqUtils import gc_fraction
import time

Entrez.email = "tryhobbituwu@gmail.com"

all_records = []

# 1. Betula platyphylla - загрузка  последовательностей
print("\n Betula platyphylla - загрузка  последовательностей...")

try:
    search_query = 'Betula platyphylla[Organism] AND chloroplast[Title]'
    """# Только хлоропластные последовательности:
search_query = 'Betula platyphylla[Organism] AND chloroplast[Title]'
# Только последовательности psbA-trnH (как в вашем примере):
search_query = 'Betula platyphylla[Organism] AND psbA-trnH[Title]'
# Только полные геномы:
search_query = 'Betula platyphylla[Organism] AND "complete genome"[Filter]'
# Только митохондриальные:
search_query = 'Betula platyphylla[Organism] AND mitochondrion[Title]'
# Без фильтра (все типы последовательностей):
search_query = 'Betula platyphylla[Organism]'"""

    handle = Entrez.esearch(
        db="nucleotide",
        term=search_query,
        retmax=5,
        sort="relevance"
    )
    betula_ids = Entrez.read(handle)["IdList"]
    handle.close()
    
    print(f"✓ Найдено записей: {len(betula_ids)}")
    
    for i, record_id in enumerate(betula_ids, 1):
        try:
            print(f"  Загрузка [{i}/{len(betula_ids)}] {record_id}...", end=" ")
            time.sleep(0.3)
            
            handle = Entrez.efetch(db="nucleotide", id=record_id, 
                                   rettype="gb", retmode="text")
            record = SeqIO.read(handle, "genbank")
            handle.close()
            record.annotations['species'] = 'Betula platyphylla'
            all_records.append(record)
            
            print(f"✓ {len(record.seq)} bp")
            
        except Exception as e:
            print(f"✗ Ошибка: {e}")
            continue
            
except Exception as e:
    print(f"✗ Ошибка поиска Betula platyphylla: {e}")


# 2. Candidatus Versatilivorator vitaminiformans - загрузка контигов
print("\n Candidatus Versatilivorator vitaminiformans - загрузка контигов...")


contig_ids = [
    "JAPVWV010000001",
    "JAPVWV010000002",
    "JAPVWV010000003",
    "JAPVWV010000004",
    "JAPVWV010000005"
]

for contig_id in contig_ids:
    try:
        print(f"  Загрузка {contig_id}...", end=" ")
        time.sleep(0.3)
        
        handle = Entrez.efetch(
            db="nucleotide", 
            id=contig_id, 
            rettype="gb", 
            retmode="text"
        )
        
        record = SeqIO.read(handle, "genbank")
        handle.close()

        gc_content = gc_fraction(record.seq)
        record.annotations['gc_content'] = gc_content
        record.annotations['species'] = 'Candidatus Versatilivorator vitaminiformans'
        all_records.append(record)
        
        print(f"✓ {len(record.seq)} bp, GC={gc_content:.4f}")
        
    except Exception as e:
        print(f"✗ Ошибка: {e}")
        continue


# 3. Сохранение всех записей в один файл GenBank

print("СОХРАНЕНИЕ РЕЗУЛЬТАТОВ")
output_file = f"good_geans.gb"
if all_records:

    with open(output_file, "w") as output_handle:
        count = SeqIO.write(all_records, output_handle, "genbank")
    print(f"\n✓ Сохранено {count} записей в файл: {output_file}")
    betula_count = sum(1 for r in all_records if r.annotations.get('species') == 'Betula platyphylla')
    versatil_count = sum(1 for r in all_records if r.annotations.get('species') == 'Candidatus Versatilivorator vitaminiformans')

    print(f"\n📋 Детали записей:")
    for i, record in enumerate(all_records, 1):
        species = record.annotations.get('species', 'Unknown')
        gc = gc_fraction(record.seq)
        print(f"  {i}. {record.id}: {len(record.seq)} bp, GC={gc:.4f} [{species}]")
        
else:
    print("✗ Не удалось загрузить ни одной последовательности!")

print("Алилуя! Файл сохранён в формате GenBank (.gb)")



✓ Найдено записей: 5
  Загрузка [1/5] 3036117715... ✓ 1320 bp
  Загрузка [2/5] 3036117714... ✓ 998 bp
  Загрузка [3/5] 3036114865... ✓ 727 bp
  Загрузка [4/5] 3036114864... ✓ 542 bp
  Загрузка [5/5] 3023078750... ✓ 459 bp

[2/2] Candidatus Versatilivorator vitaminiformans - загрузка контигов...
----------------------------------------------------------------------
  Загрузка JAPVWV010000001... ✓ 2271 bp, GC=0.6486
  Загрузка JAPVWV010000002... ✓ 4897 bp, GC=0.6620
  Загрузка JAPVWV010000003... ✓ 15926 bp, GC=0.6281
  Загрузка JAPVWV010000004... ✓ 5056 bp, GC=0.6600
  Загрузка JAPVWV010000005... ✓ 4152 bp, GC=0.6479

СОХРАНЕНИЕ РЕЗУЛЬТАТОВ

✓ Сохранено 10 записей в файл: good_geans.gb

📋 Детали записей:
  1. PV924672.1: 1320 bp, GC=0.3273 [Betula platyphylla]
  2. PV924671.1: 998 bp, GC=0.3347 [Betula platyphylla]
  3. PV921823.1: 727 bp, GC=0.4305 [Betula platyphylla]
  4. PV921822.1: 542 bp, GC=0.4354 [Betula platyphylla]
  5. PV898890.1: 459 bp, GC=0.3355 [Betula platyphylla]
  6. JA

In [2]:
#Задание 2
#GC состав
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction

# Пути к файлам
input_file = "good_geans.gb"
output_file = "good_geans_with_GC.gb"
output_file_2 = "good_geans_terminal"

# Чтение, сортировка, сохранение
records = list(SeqIO.parse(input_file, "genbank"))
records_with_gc = []
for record in records:
    gc_content = gc_fraction(record.seq)
    records_with_gc.append({
        'record': record,
        'gc': gc_content
    })

records_with_gc.sort(key=lambda x: x['gc'])
records_sorted = [item['record'] for item in records_with_gc]

with open(output_file, "w") as f:
    SeqIO.write(records_sorted, f, "genbank")

with open(output_file_2, "w") as f:
    for item in records_with_gc:
        record = item['record']
        gc = item['gc']
        line = f"{record.id}: {record.description}, GC = {gc}\n"
        print(line.strip())      
        f.write(line)   
 

print(f"✓ Отсортировано {len(records_sorted)} записей → {output_file}")

PV924672.1: UNVERIFIED: Betula platyphylla isolate XB_HM906 maturase K-like gene, partial sequence; chloroplast, GC = 0.32727272727272727
PV924671.1: UNVERIFIED: Betula platyphylla isolate XB_HM861 maturase K-like gene, partial sequence; chloroplast, GC = 0.3346693386773547
PV898890.1: Betula platyphylla isolate XB_HM906 psbA-trnH intergenic spacer region, partial sequence; chloroplast, GC = 0.3355119825708061
PV921823.1: UNVERIFIED: Betula platyphylla isolate XB_HM861 ribulose-1,5-bisphosphate carboxylase/oxygenase large subunit-like gene, partial sequence; chloroplast, GC = 0.4305364511691884
PV921822.1: UNVERIFIED: Betula platyphylla isolate XB_HM906 ribulose-1,5-bisphosphate carboxylase/oxygenase large subunit-like gene, partial sequence; chloroplast, GC = 0.4354243542435424
JAPVWV010000003.1: MAG: Candidatus Versatilivorator vitaminiformans isolate r_rhhl26 Seq3, whole genome shotgun sequence, GC = 0.6280924274770815
JAPVWV010000005.1: MAG: Candidatus Versatilivorator vitaminiform

In [4]:
# Задание 3
# трансляция последовательностей
from Bio import SeqIO

input_file = "good_geans.gb"
output_file = "good_geans_translations.txt"

records = list(SeqIO.parse(input_file, "genbank"))

with open(output_file, "w", encoding="utf-8") as f:
    total_translated = 0
    # Ищем аннотированные кодирующие области и если они есть транслируем их, иначе транслируем ВСЮ последовательность целиком
    for i, record in enumerate(records):
       
        coding_features = [
            feature for feature in record.features 
            if feature.type in ["CDS", "misc_feature", "gene"]
        ]
        
        coding_features = [
            f for f in coding_features 
            if any(key in f.qualifiers for key in ['note', 'gene', 'product', 'translation'])
        ]
        
        header = f"{record.id}: {record.description}\n"
        print(f"[{i}/{len(records)}] {record.id}")
        f.write(header)
        
        if coding_features:
            for feature in coding_features:
                total_translated += 1
                
                try:
                    location = feature.location
                    start = int(location.start) + 1
                    end = int(location.end)
                    strand = location.strand
                    strand_str = "(+)" if strand == 1 else "(-)" if strand == -1 else "(?)"
                    
                    seq = feature.extract(record.seq)
                    if strand == -1:
                        seq = seq.reverse_complement()
                    
                    protein = seq.translate(to_stop=False)
                    
                    gene = feature.qualifiers.get('gene', [''])[0] if 'gene' in feature.qualifiers else ''
                    note = feature.qualifiers.get('note', [''])[0] if 'note' in feature.qualifiers else ''
                    
                    loc_line = f"Coding sequence location = [{start}:{end}]{strand_str}\n"
                    trans_line = "Translation =\n"
                    
                    print(f"  ✓ CDS/gene: {len(protein)} а.к.")
                    
                    f.write(loc_line)
                    if gene:
                        f.write(f"gene = {gene}\n")
                    if note:
                        f.write(f"note = {note}\n")
                    f.write(trans_line)
                    f.write(f"{protein}\n\n")
                    
                except Exception as e:
                    print(f"  ✗ Ошибка: {e}")
                    continue
        else:
            total_translated += 1
            try:
                seq = record.seq
                seq_len = len(seq)
                # Проверяем, кратна ли длина 3
                if seq_len % 3 != 0:
                    print(f"  ⚠ Длина {seq_len} bp не кратна 3. Обрезаю до {seq_len - (seq_len % 3)} bp")
                    seq = seq[:seq_len - (seq_len % 3)]
                protein = seq.translate(to_stop=False)
                loc_line = f"Coding sequence location = [1:{seq_len}](+)\n"
                trans_line = "Translation =\n"
                note_line = f"note = whole sequence translation ({seq_len} bp)\n"
                print(f"  ✓ Whole sequence: {len(protein)} а.к. (из {seq_len} bp)")
                f.write(loc_line)
                f.write(note_line)
                f.write(trans_line)
                f.write(f"{protein}\n\n")
                
            except Exception as e:
                print(f"  ✗ Ошибка трансляции: {e}")
                f.write(f"  Ошибка: {e}\n\n")
                continue
        
    
    
    print(f"\n{'='*80}")
    print(f"✓ Всего записей: {len(records)}")
    print(f"✓ Транслировано: {total_translated}")
    print(f"✓ Результат в: {output_file}")
    print(f"{'='*80}")

[0/10] PV924672.1
  ✓ CDS/gene: 440 а.к.
[1/10] PV924671.1
  ✓ CDS/gene: 332 а.к.
[2/10] PV921823.1
  ✓ CDS/gene: 242 а.к.
[3/10] PV921822.1
  ✓ CDS/gene: 180 а.к.
[4/10] PV898890.1
  ✓ CDS/gene: 153 а.к.
[5/10] JAPVWV010000001.1
  ✓ Whole sequence: 757 а.к. (из 2271 bp)
[6/10] JAPVWV010000002.1
  ⚠ Длина 4897 bp не кратна 3. Обрезаю до 4896 bp
  ✓ Whole sequence: 1632 а.к. (из 4897 bp)
[7/10] JAPVWV010000003.1
  ⚠ Длина 15926 bp не кратна 3. Обрезаю до 15924 bp
  ✓ Whole sequence: 5308 а.к. (из 15926 bp)
[8/10] JAPVWV010000004.1
  ⚠ Длина 5056 bp не кратна 3. Обрезаю до 5055 bp
  ✓ Whole sequence: 1685 а.к. (из 5056 bp)
[9/10] JAPVWV010000005.1
  ✓ Whole sequence: 1384 а.к. (из 4152 bp)

✓ Всего записей: 10
✓ Транслировано: 10
✓ Результат в: good_geans_translations.txt
